In [11]:
import numpy as np
import pandas as pd
import cvxpy as cp
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.covariance import LedoitWolf
from pykalman import KalmanFilter

In [2]:
class FactorModel:
  def __init__(self, rfr, **kwargs):
    self.rfr = rfr
    super().__init__(**kwargs)

  def get_factors_exp(self, returns, factors, verbose=False):
    if isinstance(self.rfr, pd.Series):
      rf = self.rfr.reindex(returns.index)

      if rf.isna().any():
          raise ValueError("Missing risk-free observations")

      y = returns.sub(rf, axis=0)
    else:
      y = returns - float(self.rfr)

    X = sm.add_constant(
      factors.reindex(returns.index),
      has_constant="add"
    )

    rows = {}
    residuals = pd.DataFrame(
      index=returns.index,
      columns=returns.columns,
      dtype=float
    )

    for ticker in returns.columns:
      model = sm.OLS(
        y[ticker], X, missing="drop"
      ).fit()

      rows[ticker] = {
        "alpha": model.params["const"],
        "alpha_tstat": model.tvalues["const"],
        "r_squared": model.rsquared,
        **{
          f"beta_{factor}": model.params[factor]
          for factor in factors.columns
        }
      }

      residuals.loc[
        model.resid.index, ticker
      ] = model.resid

      if verbose:
        print(ticker, model.params)

    exposures = pd.DataFrame.from_dict(
      rows, orient="index"
    )

    return exposures, residuals


In [117]:
class Optimizer:
  def __init__(self, risk_aversion=2, max_exp=1.5, min_exp=1, max_w=0.20, turnover_penalty_strength=2, **kwargs):
    self.risk_aversion = float(risk_aversion)
    self.max_exp = float(max_exp)
    self.min_exp = float(min_exp)
    self.max_w = max_w
    self.turnover_penalty_strength = turnover_penalty_strength

    super().__init__(**kwargs)

  def _optimize_w(self, alphas, cov, betas, signs, borrow_rates=None, w_prev=None):
    alphas = np.asarray(alphas, dtype=float)
    cov = np.asarray(cov, dtype=float)
    betas = np.asarray(betas, dtype=float)
    signs = np.asarray(signs, dtype=float)


    cov = (cov + cov.T) / 2

    w = cp.Variable(len(alphas))

    exp_alphas = alphas @ w
    risk_term = 0.5 * self.risk_aversion * cp.quad_form(w, cp.psd_wrap(cov))

    borrow_penalty = (
        np.asarray(borrow_rates, dtype=float) @ cp.neg(w)
        if borrow_rates is not None else 0
    )

    alpha_scale = np.mean(np.abs(alphas))

    turnover_penalty = (
        self.turnover_penalty_strength * alpha_scale * cp.norm1(w - w_prev)
        if w_prev is not None else 0
    )

    signed_exposure = cp.multiply(signs, w)

    constraints = [
      signed_exposure >= 0,
      cp.sum(signed_exposure) <= self.max_exp,
      cp.sum(signed_exposure) >= self.min_exp,
      signed_exposure <= self.max_w,
      betas @ w == 0,
    ]

    prob = cp.Problem(cp.Maximize(exp_alphas - risk_term - borrow_penalty - turnover_penalty), constraints)
    prob.solve(solver=cp.CLARABEL)


    if prob.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
      raise ValueError(f"Optimization failed: {prob.status}")

    wv = w.value

    return wv

In [118]:
class KalmanAlphaEstimator:
  def __init__(self, **kwargs):
    self.A, self.C, self.Q, self.b, self.d, self.x, self.P = [None] * 7
    self.columns = None
    self.last_date = None
    super().__init__(**kwargs)

  def _ew(self, x, halflife, min_periods):
      return x.ewm(
        halflife=halflife,
        adjust=False,
        min_periods=min_periods
      ).mean()

  def _build_alpha_targets(
      self,
      returns: pd.DataFrame,
      benchmark: pd.DataFrame,
      horizon: int = 21,
      halflife: int = 126,
      min_periods: int = 126,
  ):
    mean_f = self._ew(benchmark, halflife, min_periods)
    mean_y = self._ew(returns, halflife, min_periods)
    mean_f2 = self._ew(benchmark * benchmark, halflife, min_periods)
    mean_yf = self._ew(returns.mul(benchmark, axis=0), halflife, min_periods)

    var =  mean_f2 - mean_f**2
    cov = mean_yf.sub(mean_y.mul(mean_f, axis=0))
    beta = cov.div(
      var.where(var > 1e-12),
      axis=0
    )
    observed = returns.sub(
      beta.shift(1).mul(benchmark, axis=0)
    )

    return observed

  def train(
    self,
    returns: pd.DataFrame,
    benchmark: pd.DataFrame,
    horizon: int = 21,
    halflife: int = 126,
    min_periods: int = 126,
    presistence:float=0.98
  ):
    if isinstance(benchmark, pd.DataFrame):
        benchmark = benchmark.iloc[:, 0]
    benchmark = benchmark.reindex(returns.index)

    N = len(returns.columns)
    z = self._build_alpha_targets(
      returns, benchmark, horizon, halflife, min_periods
    )
    z = z.dropna(how="any")
    self.columns = z.columns
    self.last_date = z.index[-1]

    daily_var = z.var().to_numpy()
    kf = KalmanFilter(
      transition_matrices=np.eye(N) * presistence,
      observation_matrices=np.eye(N),
      transition_offsets=np.zeros(N),
      observation_offsets=np.zeros(N),
      transition_covariance=np.diag(daily_var),
      observation_covariance=np.diag(daily_var),
      initial_state_mean=np.zeros(N),
      initial_state_covariance=np.diag(daily_var),
      em_vars=[
          "transition_covariance",
          "observation_covariance",
      ],
    )

    n_iter = 5
    kf = kf.em(z.to_numpy(), n_iter=n_iter)

    self.A = np.asarray(kf.transition_matrices)
    self.C = np.asarray(kf.observation_matrices)
    self.Q = np.asarray(kf.transition_covariance)

    self.b = np.asarray(kf.transition_offsets).reshape(-1)
    self.d = np.asarray(kf.observation_offsets).reshape(-1)

    filtered_means, filtered_covs = kf.filter(z.to_numpy())

    self.x = filtered_means[-1].copy()
    self.P = filtered_covs[-1].copy()


  def forecast(self, n_days):
    if self.x is None:
        raise RuntimeError("call train() first")

    predictions, state_covariances = [], []
    x, P = self.x.copy(), self.P.copy()

    for _ in range(n_days):
        x = self.A @ x + self.b
        P = self.A @ P @ self.A.T + self.Q
        predictions.append(self.C @ x + self.d)
        state_covariances.append(P.copy())

    idx = pd.bdate_range(self.last_date, periods=n_days + 1)[1:]

    daily_alpha = pd.DataFrame(
        np.asarray(predictions), index=idx, columns=self.columns
    )

    return (
        daily_alpha.sum(axis=0),
        daily_alpha,
        np.asarray(state_covariances),
    )

In [119]:
class Portfolio(FactorModel, Optimizer, KalmanAlphaEstimator):
  def __init__(self, universe, alpha_shrinkage=0.25, holding_days=21, **kwargs):
    self.holding_days = holding_days
    self.alpha_shrinkage = alpha_shrinkage
    self.universe = list(universe.keys())
    self.signs = list(universe.values())
    super().__init__(**kwargs)

  def get_data(self, start="2020-01-01", end="2026-10-01", benchmark="^GSPC"):
    prices = yf.download(self.universe + [benchmark], start=start, end=end)["Close"]
    rets = prices.pct_change(fill_method=None).dropna()

    return rets[self.universe], rets[[benchmark]]

  def _get_factor_exposures(self, returns, benchmark):
    factors = (benchmark.sub(self.rfr, axis=0)
               if hasattr(self.rfr, "index") else benchmark - self.rfr)
    exposures, residuals = self.get_factors_exp(returns, factors)

    alphas = exposures["alpha"]
    B = exposures.filter(like="beta_")

    return alphas, B, residuals

  def _rf(self, index):
    if isinstance(self.rfr, pd.Series):
        rf = self.rfr.reindex(index)
        if rf.isna().any():
            raise ValueError("Missing risk-free data")
        return rf
    return pd.Series(float(self.rfr), index=index)

  def get_portfolio(self, returns, benchmark, borrow_rates_annual=None, previous_w=None):
    assets = self.universe
    R = returns.loc[:, assets]
    M = benchmark.reindex(R.index)
    valid = R.notna().all(axis=1) & M.notna().all(axis=1)
    R, M = R.loc[valid], M.loc[valid]

    rf = self._rf(R.index)
    R_ex = R.sub(rf, axis=0)
    F_ex = M.sub(rf, axis=0)

    exposures, residuals = self.get_factors_exp(R, F_ex)
    exposures = exposures.loc[assets]
    residuals = residuals.loc[:, assets].dropna()
    if len(residuals) < max(60, 3 * len(assets)):
        raise ValueError("Insufficient observations")

    self.train(R_ex, F_ex)
    alpha_kf, _, _ = self.forecast(self.holding_days)
    alpha_h = alpha_kf.reindex(assets)
    if alpha_h.isna().any():
        raise ValueError("Kalman alpha missing for some assets")
    alpha_h = alpha_h.to_numpy() * self.alpha_shrinkage

    B = exposures.filter(regex=r"^beta_").to_numpy().T      #
    cov_h = LedoitWolf(store_precision=False).fit(residuals.to_numpy()).covariance_ * self.holding_days

    borrow_h = np.zeros(len(assets))
    if borrow_rates_annual is not None:
        annual = pd.Series(borrow_rates_annual).reindex(assets)
        if annual.isna().any():
            raise ValueError("Missing borrow rate")
        borrow_h = annual.to_numpy(dtype=float) * self.holding_days / 252

    prev = np.zeros(len(assets)) if previous_w is None else previous_w.reindex(assets).fillna(0).to_numpy()

    w = self._optimize_w(alphas=alpha_h, cov=cov_h, betas=B,
                         signs=np.asarray(self.signs),
                         borrow_rates=borrow_h, w_prev=prev)
    return pd.Series(w, index=assets, name="weight")

In [120]:
def rolling_backtest(ptf, returns, benchmark, lookback=504, rebalance=63,
                     cost_bps=5, periods=252):
    cost = cost_bps / 1e4
    w_cur = pd.Series(0.0, index=returns.columns)
    port, turnover, weights = {}, {}, {}
    w_prev = None

    for start in range(lookback, len(returns), rebalance):
        train_r = returns.iloc[start - lookback:start]
        train_b = benchmark.iloc[start - lookback:start]
        hold = returns.iloc[start:start + rebalance]

        try:
            w_new = ptf.get_portfolio(train_r, train_b, previous_w=w_prev)
        except ValueError as e:
            print(f"{returns.index[start].date()}: {e}; keeping previous weights")
            w_new = w_cur

        turnover[returns.index[start]] = (w_new - w_cur).abs().sum()
        weights[returns.index[start]] = w_new
        w = w_new.copy()

        for i, (d, r) in enumerate(hold.iterrows()):
            pr = float((w * r).sum())
            if i == 0:
                pr -= cost * turnover[returns.index[start]]
            port[d] = pr
            w = w * (1 + r) / (1 + pr)
        w_cur = w
        w_prev = w

    port = pd.Series(port, name="strategy")
    bench = benchmark.iloc[:, 0].reindex(port.index)

    X = np.column_stack([np.ones(len(port)), bench.values])
    alpha_d, beta = np.linalg.lstsq(X, port.values, rcond=None)[0]
    equity = (1 + port).cumprod()

    stats = pd.Series({
        "ann_return": (1 + port).prod() ** (periods / len(port)) - 1,
        "ann_vol": port.std() * np.sqrt(periods),
        "sharpe": port.mean() / port.std() * np.sqrt(periods),
        "max_drawdown": (equity / equity.cummax() - 1).min(),
        "realized_beta": beta,
        "ann_alpha": alpha_d * periods,
        "avg_turnover_per_rebal": np.mean(list(turnover.values())),
        "bench_ann_return": (1 + bench).prod() ** (periods / len(bench)) - 1,
    })

    return {
        "returns": port,
        "equity": equity,
        "weights": pd.DataFrame(weights).T,
        "turnover": pd.Series(turnover),
        "stats": stats,
    }


def plot_backtest(res, benchmark):
    eq = res["equity"]
    bench_eq = (1 + benchmark.iloc[:, 0].reindex(eq.index)).cumprod()
    fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    eq.plot(ax=ax[0], label="long/short")
    bench_eq.plot(ax=ax[0], label="benchmark", alpha=0.7)
    ax[0].legend(); ax[0].set_title("Equity curve")
    (eq / eq.cummax() - 1).plot(ax=ax[1], color="firebrick")
    ax[1].set_title("Drawdown")
    plt.tight_layout(); plt.show()